# Small regressions — linear & logistic

Runs the four PDMP samplers (Boom, Sticky-Boom, ZZ, Sticky-ZZ) plus
Gaussian-prior NUTS on either:

- a small **linear regression** (D=10, 3 signals), or
- a small **logistic regression** (D=10, 3 signals).

Imports building blocks from `sazz.scripts.linear_regression` /
`sazz.scripts.logistic_regression` so the runs happen in-memory — nothing
is read or written to disk. Switch the `TASK` flag below to flip between
them.

If you want sparse high-D experiments, see `sparse_results.ipynb` (loads
disk-persisted runs from `linear_regression.py --save`).

## 0. Setup

In [ ]:
import os, sys
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import numpy as np
import torch
import matplotlib.pyplot as plt

torch.set_default_dtype(torch.float64)

# ---- Pick the task ----
TASK = "logistic"          # "linear" or "logistic"
INCLUDE_HORSESHOE = True    # add horseshoe NUTS (slow)

if TASK == "linear":
    from sazz.scripts.linear_regression import (
        Config, set_seed, make_data, analytic_posterior,
        run_pdmps, run_nuts, compute_metrics, print_table,
    )
    from sazz.models.make_models import make_linear_regression as make_target
    cfg = Config(N=200, D=9, n_signals=3, signal_scale=1.5,
                 intercept_true=0.5, noise_std=0.3,
                 prior_std=1.0, lik_noise_std=0.5, seed=0, thinning="pli")
elif TASK == "logistic":
    from sazz.scripts.logistic_regression import (
        Config, set_seed, make_data,
        run_pdmps, run_nuts, compute_metrics, print_table,
    )
    from sazz.models.make_models import make_logistic_regression as make_target
    cfg = Config(N=300, D=9, n_signals=3, signal_scale=1.5,
                 intercept_true=0.5, prior_std=1.0, seed=0, thinning="pli")
else:
    raise ValueError(TASK)

print(f"task={TASK}  N={cfg.N}  D={cfg.D}  K={cfg.n_signals}  seed={cfg.seed}")

## 1. Data + held-out test set

Generated synthetically from the same DGP for train and test. For
logistic, the test labels are fresh Bernoulli draws under the true
coefficients.

In [ ]:
set_seed(cfg.seed)
X, y, true_coefs, is_signal = make_data(cfg)
coef_names = ["intercept"] + [f"β_{i}" for i in range(cfg.D)]

# Held-out test set, fresh DGP draw — same shape as training X (D covariates).
test_rng = np.random.default_rng(cfg.seed + 1)
X_te = test_rng.normal(size=(cfg.N, cfg.D))
X_te = (X_te - X_te.mean(0)) / X_te.std(0)

beta_te = true_coefs[1:]      # slopes
int_te  = true_coefs[0]       # intercept

if TASK == "linear":
    y_te = X_te @ beta_te + int_te                                  # noise-free
else:
    logits = X_te @ beta_te + int_te
    y_te   = test_rng.binomial(1, 1.0 / (1.0 + np.exp(-logits)))

X_te_aug = np.column_stack([np.ones(cfg.N), X_te])                  # always

print(f"signals = {int(is_signal.sum())}/{len(true_coefs)}")
print(f"true coefs: {dict(zip(coef_names, np.round(true_coefs, 3)))}")

## 2. Run all samplers

Builds the PDMP target, runs all four PDMP samplers, then Gaussian-prior
NUTS (and horseshoe NUTS if you flipped the toggle). Everything stays
in-memory; no `.pt` files are written.

In [ ]:
rows: list[dict] = []

# Closed-form posterior — Gaussian prior + Gaussian likelihood, with the
# intercept as coordinate 0 of X_aug (its own prior precision).
# Logistic has no closed form (Bernoulli likelihood is non-conjugate).
if TASK == "linear":
    X_aug = np.column_stack([np.ones(cfg.N), X])
    mu_an, Sigma_an = analytic_posterior(X_aug, y, cfg)
    an_samples = np.random.default_rng(cfg.seed).multivariate_normal(
        mu_an, Sigma_an, size=cfg.nuts_draws * cfg.nuts_chains)
    rows.append(dict(name="Analytic", samples=an_samples, wall=None,
                     sticky=False, color="k", marker="D"))

print("Running Gaussian-prior NUTS...")
rows.append(run_nuts(X, y, cfg, prior="gaussian"))

if INCLUDE_HORSESHOE:
    print("Running horseshoe NUTS (slow)...")
    rows.append(run_nuts(X, y, cfg, prior="horseshoe",
                         k_signals_guess=cfg.n_signals))

if TASK == "linear":
    target = make_target(
        torch.tensor(X), torch.tensor(y),
        prior_std=cfg.prior_std, intercept_prior_std=cfg.intercept_prior_std,
        noise_std=cfg.lik_noise_std, diagonal_only=False,
    )
else:
    target = make_target(
        torch.tensor(X), torch.tensor(y, dtype=torch.long),
        prior_std=cfg.prior_std, intercept_prior_std=cfg.intercept_prior_std,
        diagonal_only=False,
    )
print("\nRunning PDMP samplers...")
rows.extend(run_pdmps(target, cfg))

print(f"\nLoaded {len(rows)} methods: {[r['name'] for r in rows]}")

## 3. Metrics table

In [ ]:
sd_ref = next(
    (r["samples"].std(0) for r in rows if r["name"] in ("NUTS-Gaussian", "Analytic")),
    None,
)

rows = [compute_metrics(r, true_coefs, is_signal, X_te_aug, y_te, sd_ref)
        for r in rows]
print_table(rows)

## 4. Coefficient calibration plot

Errorbars (μ ± 2σ) for each method on each coordinate, with the true
value marked. Yellow stripes mark true signals.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4.5))
idx = np.arange(len(true_coefs))
offsets = np.linspace(-0.32, 0.32, len(rows))

for r, off in zip(rows, offsets):
    mu = r["samples"].mean(0); sd = r["samples"].std(0)
    ax.errorbar(idx + off, mu, yerr=2 * sd, fmt=r["marker"], color=r["color"],
                label=r["name"], capsize=2, markersize=4, lw=1)

ax.scatter(idx, true_coefs, marker="x", color="k", s=70, linewidths=2,
           label="true", zorder=5)
for i in np.where(is_signal)[0]:
    ax.axvspan(i - 0.45, i + 0.45, color="gold", alpha=0.12)
ax.axhline(0, color="grey", lw=0.5)
ax.set_xticks(idx); ax.set_xticklabels(coef_names)
ax.set_ylabel("coefficient")
ax.set_title(f"{TASK} regression — posterior μ ± 2σ ({cfg.thinning})")
ax.legend(fontsize=8, ncol=3, frameon=False)
plt.tight_layout(); plt.show()

## 5. Trace plots — one signal coord + one null coord

Eyeball check that all chains have settled and are mixing on the same
posterior. The signal coordinate is the one with the largest true
magnitude; the null is the smallest non-intercept coordinate.

In [ ]:
def trace_grid(coord_idx: int, ylabel: str):
    fig, axes = plt.subplots(1, len(rows), figsize=(3.0 * len(rows), 2.8),
                             sharey=True)
    if len(rows) == 1:
        axes = [axes]
    for ax, r in zip(axes, rows):
        ax.plot(r["samples"][:, coord_idx], lw=0.3, color=r["color"])
        ax.axhline(true_coefs[coord_idx], color="k", lw=1, ls="--",
                   label=f"true = {true_coefs[coord_idx]:.2f}")
        ax.set_title(r["name"], fontsize=9)
        ax.set_xlabel("sample idx")
        ax.legend(loc="best", fontsize=7, frameon=False)
    axes[0].set_ylabel(ylabel)
    plt.tight_layout(); plt.show()


j_signal = int(np.argmax(np.abs(true_coefs)))
# pick smallest non-intercept; if no intercept just smallest
start = 0
j_null = int(np.argmin(np.abs(true_coefs[start:])) + start)

trace_grid(j_signal, f"{coef_names[j_signal]} (signal)")
trace_grid(j_null,   f"{coef_names[j_null]} (null)")

## 6. ESS per coordinate

Initial-positive-sequence ESS (Geyer's IPS truncation), computed
per-coordinate. For sticky samplers we also compute:

- **active-ESS:** ESS conditional on the coordinate being non-zero. Tells
  you about within-slab mixing.
- **indicator-ESS:** ESS of the binary inclusion indicator
  γ_i = 𝟙[β_i ≠ 0]. Tells you how well the chain mixes the
  spike-vs-slab decision.

In [ ]:
def ess_ips(samples: np.ndarray, max_lag: int | None = None) -> np.ndarray:
    """Geyer's initial-positive-sequence ESS, vectorised over coordinates."""
    x = samples - samples.mean(0, keepdims=True)
    n, d = x.shape
    var = (x ** 2).mean(0)
    if max_lag is None:
        max_lag = min(n - 1, 1000)
    rho_sum = np.zeros(d)
    prev_pair = np.full(d, np.inf)
    active = np.ones(d, dtype=bool)
    k = 1
    while k + 1 <= max_lag:
        c_k   = (x[:n - k]     * x[k:]).mean(0)     / np.maximum(var, 1e-30)
        c_kp1 = (x[:n - k - 1] * x[k + 1:]).mean(0) / np.maximum(var, 1e-30)
        pair = c_k + c_kp1
        kill = active & ((pair <= 0) | (pair >= prev_pair))
        active = active & ~kill
        rho_sum = rho_sum + np.where(active, pair, 0.0)
        prev_pair = np.where(active, pair, prev_pair)
        if not active.any():
            break
        k += 2
    tau = 1.0 + 2.0 * rho_sum
    return n / np.clip(tau, 1.0, None)


print(f"{'sampler':<16}  {'metric':<10}  {'min':>8}  {'median':>8}  {'min/sec':>10}")
print("-" * 60)
for r in rows:
    s = r["samples"]
    wall = r.get("wall")
    if r["sticky"]:
        # Active ESS (within-slab mixing)
        active = np.abs(s) > 1e-8
        ess_a = np.full(s.shape[1], np.nan)
        for d in range(s.shape[1]):
            sub = s[active[:, d], d]
            if len(sub) >= 50:
                ess_a[d] = ess_ips(sub[:, None]).item()
        # Indicator ESS (spike-slab mixing)
        ind = active.astype(float)
        ess_i = ess_ips(ind)
        for label, ess in [("active", ess_a), ("indicator", ess_i)]:
            mn, md_ = np.nanmin(ess), np.nanmedian(ess)
            ps = f"{mn / wall:>10.1f}" if wall else f"{'—':>10}"
            print(f"{r['name']:<16}  {label:<10}  {mn:>8.0f}  {md_:>8.0f}  {ps}")
    else:
        ess = ess_ips(s)
        mn, md_ = np.min(ess), np.median(ess)
        ps = f"{mn / wall:>10.1f}" if wall else f"{'—':>10}"
        print(f"{r['name']:<16}  {'ESS':<10}  {mn:>8.0f}  {md_:>8.0f}  {ps}")